In [21]:
import os
import re

import pypdf
import anthropic
from langchain.chains import RetrievalQA

In [22]:
# Basic setup/config items
DOC_DIR = "../documents/Credit"
FILE_NAME = "sec.gov_Archives_edgar_data_1701758_000121390018004741_fs12018ex10-1_thelovesac.htm.pdf"
# LLM_MODEL_NAME = "claude-3-5-sonnet-20241022"
LLM_MODEL_NAME = "claude-opus-4-20250514"

In [23]:
# https://www.sec.gov/Archives/edgar/data/1701758/000121390018004741/fs12018ex10-1_thelovesac.htm
details_to_extract = [
    'Parties involved (lenders, agent, borrower)',
    'Parties details (address, business, description)',
    'Terms (interest charged, rates, fees, payments, liens)',
    'Loan details (credit limit, letter of credit, repayments, timeline)'
    'Payments (agent clawback, sharing of payments, settlement among lenders)',
    'Miscellaneous (litigation, insurance, taxes)'
]

In [24]:
# Function to split the document into chunks and embed them
def read_pdf(doc_path):
    reader = pypdf.PdfReader(doc_path)
    text = "\n".join([page.extract_text() for page in reader.pages])
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text)
    # Remove page numbers
    text = re.sub(r'\n\s*\d+\s*\n', '\n', text)

    return text

In [25]:
def get_llm_text(pdf_file):
    reader = pypdf.PdfReader(pdf_file)
    text = "\n".join([page.extract_text() for page in reader.pages])

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text)

    # Remove page numbers
    text = re.sub(r'\n\s*\d+\s*\n', '\n', text)
    return text

In [16]:
pdf = (os.path.join(DOC_DIR, FILE_NAME))
document_text = get_llm_text(pdf)
print(document_text[:50])

EX-10.1 5 fs12018ex10-1_thelovesac.htm WELLS FARGO CREDIT AGREEMENT Exhibit 10.1 [EXECUTION] CREDIT AGREEMENT Dated as of February 2, 2018 among THE LOVESAC COMPANY as the Lead Borrower SAC ACQUISITION LLC as Guarantor WELLS FARGO BANK, NATIONAL ASSOCIATION as Agent, L/C Issuer and Swing Line Lender, and The Other Lenders Party Hereto WELLS FARGO BANK, NATIONAL ASSOCIATION, as Sole Lead Arranger and Sole Bookrunner 6/25/25, 2:49 PM sec.gov/Archives/edgar/data/1701758/000121390018004741/fs12018ex10-1_thelovesac.htm https://www.sec.gov/Archives/edgar/data/1701758/000121390018004741/fs12018ex10-1_thelovesac.htm 1/129 TABLE OF CONTENTS Page ARTICLE I DEFINITIONS AND ACCOUNTING TERMS 1 1.01 Defined Terms 1 1.02 Other Interpretive Provisions 43 1.03 Accounting Terms 44 1.04 Rounding 44 1.05 Times of Day 44 1.06 Letter of Credit Amounts 44 ARTICLE II THE COMMITMENTS AND CREDIT EXTENSIONS 44 2.01 Committed Loans; Reserves 44 2.02 Borrowings, Conversions and Continuations of Committed Loans 45 

In [26]:
# Initialize the Anthropic client
client = anthropic.Anthropic()

def summarize_document(text, details_to_extract, model=LLM_MODEL_NAME, max_tokens=1000):
    # Format the details to extract to be placed within the prompt's context
    details_to_extract_str = '\n'.join(details_to_extract)

    # Prompt the model to summarize the sublease agreement
    prompt = f"""Summarize the following credit agreement. Focus on these key aspects:

    {details_to_extract_str}

    Provide the summary in bullet points nested within the XML header for each section. For example:

    <parties involved>
    - Lender: [Name]
    // Add more details as needed
    </parties involved>

    If any information is not explicitly stated in the document, note it as "Not specified". Do not preamble.

    Sublease agreement text:
    {text}
    """

    response = client.messages.create(
        model=model,
        max_tokens=max_tokens,
        system="You are a legal analyst specializing in credit law, known for highly accurate and detailed summaries of credit agreements.",
        messages=[
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": "Here is the summary of the credit agreement: <summary>"}
        ],
        stop_sequences=["</summary>"]
    )

    return response.content[0].text


sublease_summary = summarize_document(document_text, details_to_extract)
print(sublease_summary)



<parties involved>
- Lender: Wells Fargo Bank, National Association (as Agent, L/C Issuer, Swing Line Lender, Sole Lead Arranger and Sole Bookrunner)
- Lead Borrower: The Lovesac Company (a Delaware corporation)
- Guarantor: SAC Acquisition LLC (a Delaware limited liability company, also referred to as "Parent")
- Additional Borrowers: May be added as parties after the closing date
- Additional Guarantors: May be added as parties after the closing date
</parties involved>

<parties details>
- Wells Fargo Bank, National Association: National banking association serving multiple roles including Agent, Lender, L/C Issuer, and Swing Line Lender
- The Lovesac Company: Delaware corporation, serves as Lead Borrower and agent for all Borrowers
- SAC Acquisition LLC: Delaware limited liability company, Parent entity providing guarantee
- Address details: Not specified in the document for the parties
- Business description: The Lovesac Company appears to be a retail business with inventory, st